# expdpy — Explore panel data

_Notebook version: built 2026-06-22 03:27 UTC — re-open this notebook from GitHub if yours is older, to get the latest version._

A cloud-runnable walkthrough of the **Explore** module of [expdpy](https://github.com/cmg777/expdpy) on the bundled `kuznets` panel. Run the install cell below first, then run the rest top to bottom.

> If Colab prompts you to **restart the runtime** after the install, do so, then continue from the setup cell.

This notebook mirrors the [Explore page](https://cmg777.github.io/expdpy/explore.html) of the docs.

In [ ]:
!pip install -q "expdpy @ git+https://github.com/cmg777/expdpy.git" "numba>=0.61"
!pip install -q --force-reinstall --no-deps "expdpy @ git+https://github.com/cmg777/expdpy.git"

In [ ]:
# Ensure Plotly figures render in Google Colab (a no-op in other notebook frontends).
import plotly.io as pio

try:
    import google.colab  # noqa: F401  (present only on Colab)

    pio.renderers.default = "colab"
except ImportError:
    pass

The **Explore** module is the first look at your data: descriptive and correlation tables,
distributions, missing-value maps, time trends, group comparisons and scatter plots. Every
function takes a `pandas` DataFrame and returns an interactive
[Plotly](https://plotly.com/python/) figure or a publication-quality
[Great Tables](https://posit-dev.github.io/great-tables/) object.

The lead dataset throughout is the bundled `kuznets` panel (80 countries × 2015–2025) — a
synthetic dataset whose regional inequality (`gini_regional`) traces an **N-shaped Kuznets
curve** in income.

In [ ]:
import numpy as np

import expdpy as ex
from expdpy.data import load_kuznets, load_kuznets_data_def

# `set_labels` attaches the data dictionary's human-readable labels, so every figure and
# table below titles its axes and headers with full labels such as
# "Regional inequality (Gini)" rather than the bare column name.
df = ex.set_labels(load_kuznets(), load_kuznets_data_def())
df[["country", "continent", "year", "gini_regional", "gdp_pc", "log_gdp_pc"]].head()

## Guided tour

### Treat outliers

Winsorize the (skewed) numeric variables at the 1st/99th percentile:

In [ ]:
analysis = ex.treat_outliers(
    df[["gini_regional", "gdp_pc", "school_enrollment", "resource_rents"]],
    percentile=0.01,
).dropna()
analysis.describe().round(3)

### Describe the data

In [ ]:
ex.explore_descriptive_table(analysis).gt

`explore_histogram` summarizes a numeric variable; `explore_bar_plot` counts the levels of a
categorical one:

In [ ]:
ex.explore_histogram(df, "gini_regional").fig

In [ ]:
ex.explore_bar_plot(df, "continent").fig

A heatmap of where data is missing across the panel — essential before you model:

In [ ]:
ex.explore_missing_values_plot(df, time="year").fig

### Relationships between variables

Pearson correlations appear above, Spearman below the diagonal; significant cells are bold.

In [ ]:
ex.explore_correlation_table(analysis).gt

In [ ]:
ex.explore_correlation_plot(analysis).fig

The most extreme observations of a variable:

In [ ]:
ex.explore_ext_obs_table(
    df, n=5, entity=["country"], time="year", var="gini_regional"
).gt

### Trends over time

In [ ]:
ex.explore_trend_plot(df, time="year", var=["gini_regional"]).fig

In [ ]:
ex.explore_quantile_trend_plot(df, time="year", var="gini_regional").fig

### Comparing groups

In [ ]:
ex.explore_bar_plot_by_group(df, "continent", "gini_regional", np.nanmedian).fig

In [ ]:
ex.explore_violin_plot_by_group(df, "continent", "gini_regional").fig

In [ ]:
ex.explore_trend_plot_by_group(
    df, time="year", group_var="continent", var="gini_regional"
).fig

### Scatter plot with LOESS

The **N-shaped Kuznets curve** — regional inequality against (log) GDP per capita, sized by
population and colored by continent. The LOESS smoother traces the rise–fall–rise:

In [ ]:
ex.explore_scatter_plot(
    df, x="log_gdp_pc", y="gini_regional", color="continent", size="population", loess=1
).fig

## Examples

A **basic** (minimal call) and **advanced** (full options) example for every Explore
function.

### `treat_outliers`

Basic — winsorize a single variable at the 1st/99th percentile:

In [ ]:
ex.treat_outliers(df["gdp_pc"], percentile=0.01).describe()

Advanced — winsorize several columns at the 5th/95th percentile, with cut-offs computed
within each continent:

In [ ]:
ex.treat_outliers(
    df[["gini_regional", "gdp_pc"]], percentile=0.05, by=df["continent"]
).describe()

### `explore_descriptive_table`

Basic:

In [ ]:
ex.explore_descriptive_table(df).gt

Advanced — set the decimals per statistic (`None` drops that column) and a caption:

In [ ]:
ex.explore_descriptive_table(
    df,
    digits=(0, 2, 2, None, None, 2, None, None),
    caption="Kuznets panel",
).gt

### `explore_correlation_table`

Basic:

In [ ]:
ex.explore_correlation_table(df[["gini_regional", "gdp_pc", "log_gdp_pc"]]).gt

Advanced — more decimals, a stricter bold threshold, and a caption:

In [ ]:
ex.explore_correlation_table(
    df[["gini_regional", "gdp_pc", "log_gdp_pc", "trade_share"]],
    digits=3,
    bold=0.01,
    caption="Correlations (kuznets)",
).gt

### `explore_ext_obs_table`

Basic:

In [ ]:
ex.explore_ext_obs_table(df, n=5).gt

Advanced — the ten most extreme observations of a chosen variable:

In [ ]:
ex.explore_ext_obs_table(
    df, n=10, entity=["country"], time="year", var="gini_regional"
).gt

### `explore_correlation_plot`

Basic — a correlation heatmap:

In [ ]:
ex.explore_correlation_plot(df[["gini_regional", "gdp_pc", "log_gdp_pc"]]).fig

Advanced — the ellipse style (R `corrplot` look):

In [ ]:
ex.explore_correlation_plot(
    df[["gini_regional", "gdp_pc", "log_gdp_pc", "trade_share"]],
    style="ellipse",
).fig

### `explore_trend_plot`

Basic — the mean of one variable over time, with standard-error bars:

In [ ]:
ex.explore_trend_plot(df, time="year", var=["gini_regional"]).fig

Advanced — several variables on one chart:

In [ ]:
ex.explore_trend_plot(df, time="year", var=["gini_regional", "trade_share"]).fig

### `explore_quantile_trend_plot`

Basic — the default quantiles of a variable over time:

In [ ]:
ex.explore_quantile_trend_plot(df, time="year", var="gini_regional").fig

Advanced — custom quantile levels and no per-observation points:

In [ ]:
ex.explore_quantile_trend_plot(
    df, time="year", var="gini_regional", quantiles=(0.1, 0.5, 0.9), points=False
).fig

### `explore_bar_plot_by_group`

Basic — the group mean of a variable:

In [ ]:
ex.explore_bar_plot_by_group(df, "continent", "gini_regional").fig

Advanced — a different statistic, bars ordered by it, and a custom color:

In [ ]:
ex.explore_bar_plot_by_group(
    df,
    "continent",
    "gini_regional",
    stat_fun=np.nanmedian,
    order_by_stat=True,
    color="#4682b4",
).fig

### `explore_trend_plot_by_group`

Basic — one line per group over time:

In [ ]:
ex.explore_trend_plot_by_group(
    df, time="year", group_var="continent", var="gini_regional"
).fig

Advanced — add standard-error bars and drop the markers:

In [ ]:
ex.explore_trend_plot_by_group(
    df,
    time="year",
    group_var="continent",
    var="gini_regional",
    error_bars=True,
    points=False,
).fig

### `explore_violin_plot_by_group`

Basic — the distribution of a variable across groups:

In [ ]:
ex.explore_violin_plot_by_group(df, "continent", "gini_regional").fig

Advanced — order groups by mean and orient the violins vertically:

In [ ]:
ex.explore_violin_plot_by_group(
    df, "continent", "gini_regional", order_by_mean=True, group_on_y=False
).fig

### `explore_histogram`

Basic — a 30-bin histogram:

In [ ]:
ex.explore_histogram(df, "gini_regional").fig

Advanced — finer bins on another variable:

In [ ]:
ex.explore_histogram(df, "gdp_pc", bins=50).fig

### `explore_bar_plot`

Basic — category counts:

In [ ]:
ex.explore_bar_plot(df, "continent").fig

Advanced — order bars by descending count and set a custom color:

In [ ]:
ex.explore_bar_plot(df, "continent", order_by_count=True, color="red").fig

### `explore_missing_values_plot`

Basic — the fraction of missing values by variable and year:

In [ ]:
ex.explore_missing_values_plot(df, time="year").fig

Advanced — restrict to numeric variables and show only whether values are missing:

In [ ]:
ex.explore_missing_values_plot(df, time="year", no_factors=True, binary=True).fig

### `explore_scatter_plot`

Basic — a plain scatter of two variables:

In [ ]:
ex.explore_scatter_plot(df, x="log_gdp_pc", y="gini_regional").fig

Advanced — map color and size to other columns and add a size-weighted LOESS smoother:

In [ ]:
ex.explore_scatter_plot(
    df,
    x="log_gdp_pc",
    y="gini_regional",
    color="continent",
    size="population",
    loess=2,
    alpha=0.6,
).fig

## Where to go next

- [Analyze panel data](analyze.qmd) — turn these patterns into estimates (fixed effects,
  random effects, CRE, event studies).
- [Learn panel data](learn.qmd) — the ideas behind the methods, with runnable sandboxes.
- Prefer no code? Launch the [Explore app](streamlit.qmd).